# Proyecto Final — Etapa 2: Clasificación de Textos por Década

**Objetivo:** predecir la década de origen (150–188) de un párrafo de texto histórico en español.

**Enfoque:** fine-tuning de **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), modelo BERT preentrenado sobre texto en español con whole-word masking, adaptado a 39 clases.

**Transferencia de aprendizaje:** se parte de los pesos del modelo preentrenado y se ajustan todos los parámetros en el downstream task de clasificación.

**Datos:** únicamente el conjunto de entrenamiento oficial. No se usaron datos de prueba durante el entrenamiento. La separación tren/validación es estricta (split 90/10 estratificado).

## 1. Instalación y librerías

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.optim import AdamW

## 2. Configuración

In [ ]:
IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LEN    = 256
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
SEED       = 42
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 3. Datos

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)

print(f"Train: {train_df.shape}  |  Eval: {eval_df.shape}")
print(f"Décadas únicas: {sorted(train_df['decade'].unique())}")
print(f"Distribución (primeras 5 décadas):\n{train_df['decade'].value_counts().sort_index().head()}")
train_df.head(3)

## 4. Preprocesamiento

**Label encoding** de las 39 décadas y **split 90/10 estratificado** para validación.

In [ ]:
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)
print(f"Clases: {NUM_CLASSES}")

train_data, val_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["label"],
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

## 5. Tokenización y Dataset

**BETO Tokenizer** con `max_length=256`, padding y truncamiento. El `Dataset` devuelve `input_ids`, `attention_mask` y opcionalmente `labels`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


train_loader = DataLoader(
    DecadeDataset(train_data["text"], train_data["label"]),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    DecadeDataset(val_data["text"], val_data["label"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=True,
)
eval_loader = DataLoader(
    DecadeDataset(eval_df["text"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=True,
)
print("DataLoaders listos")

## 6. Modelo

**BETO** preentrenado en español con whole-word masking. Se añade una **capa de clasificación lineal** sobre el token `[CLS]` para las 39 clases.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales:     {total_p:,}")
print(f"Parámetros entrenables: {trainable_p:,}")

## 7. Entrenamiento

**AdamW** con weight decay 0.01, **warmup lineal** del 10% y gradient clipping en 1.0. Se guarda el **mejor checkpoint** según `val_acc`.

In [ ]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            if training:
                optimizer.zero_grad()
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += out.loss.item()
            preds       = out.logits.argmax(-1)
            correct    += (preds == lbls).sum().item()
            n          += len(lbls)
    return total_loss / len(loader), correct / n


best_val_acc = 0.0
best_ckpt    = os.path.join(OUT_DIR, "best_beto.pt")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    va_loss, va_acc = run_epoch(val_loader,   training=False)
    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
        f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}"
    )
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f"  → Checkpoint guardado (val_acc={va_acc:.4f})")

print(f"\nMejor val_acc: {best_val_acc:.4f}")

## 8. Predicción y Envío

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in eval_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(-1).cpu().numpy())

submission = pd.DataFrame({
    "id":     eval_df["id"],
    "answer": le.inverse_transform(all_preds),
})

sub_path = os.path.join(OUT_DIR, "submission_beto_etapa2.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"Distribución predicciones:\n{submission['answer'].value_counts().sort_index()}")
submission.head(10)